# 0. 사전준비

In [1]:
# 1. cuda-toolkit 13.0 설치 완료 (nvcc --version)
# 2. pyproject.toml에 있는 transformers, dataset 지우기 (uv remove transformers datasets)
# 3. pyproject.toml파일 업데이트:
# ```
# [tool.uv.sources]
# torch = { index = "pytorch-cu130" }
# torchvision = { index = "pytorch-cu130" }
# unsloth = { git = "https://github.com/unslothai/unsloth.git" }
# unsloth-zoo = { git = "https://github.com/unslothai/unsloth-zoo.git" }

# [[tool.uv.index]]
# name = "pytorch-cu130"
# url = "https://download.pytorch.org/whl/cu130"
# explicit = true
# ```

# 4. unsloth 설치 (uv add unsloth unsloth-zoo)
# 5. triton 설치 (uv pip install https://github.com/woct0rdho/triton-windows/releases/download/v3.2.0-windows.post1/triton-3.2.0-cp311-cp311-win_amd64.whl --no-deps)

# 1. 모델 불러오기

## 1) 베이스 모델 불러오기

In [1]:
import torch

print("torch version       :", torch.__version__)
print("torch.version.cuda  :", torch.version.cuda)
print("cuda available      :", torch.cuda.is_available())
print("device count        :", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu name            :", torch.cuda.get_device_name(0))

torch version       : 2.10.0+cu130
torch.version.cuda  : 13.0
cuda available      : True
device count        : 1
gpu name            : NVIDIA GeForce RTX 4070 Laptop GPU


In [3]:
import triton
import inspect

print("triton file:", triton.__file__)
print("triton version:", getattr(triton, "__version__", "unknown"))

import triton.runtime.jit as tj
print("jit file:", tj.__file__)
print("has constexpr_function:", hasattr(tj, "constexpr_function"))
print([x for x in dir(tj) if "constexpr" in x.lower()])

triton file: c:\potenup3\03-utilizing-huggingface\.venv\Lib\site-packages\triton\__init__.py
triton version: 3.2.0
jit file: c:\potenup3\03-utilizing-huggingface\.venv\Lib\site-packages\triton\runtime\jit.py
has constexpr_function: False
[]


In [3]:
from unsloth import FastLanguageModel 
import torch

max_seq_length = 1024    # 한 번에 처리할 최대 토큰 수 (클수록 VRAM 많이 사용)
dtype = None             # GPU에 맞게 자동 선택 (None 권장)
load_in_4bit = True      # 4bit 압축 로드 여부 (True = VRAM 절약)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

print(f"✅ 모델 로드 완료!")
print(f"   dtype : {next(model.parameters()).dtype}")
print(f"   device: {next(model.parameters()).device}")

==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4070 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ 모델 로드 완료!
   dtype : torch.bfloat16
   device: cuda:0


## 2) LoRA 어댑터 준비

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # LoRA 크기 (클수록 학습 많이, VRAM 많이 사용)
    target_modules = [           # 어댑터 붙일 레이어 선택
        "q_proj", "k_proj",      #   └─ 어텐션 레이어
        "v_proj", "o_proj",
        "gate_proj", "up_proj",  #   └─ MLP 레이어
        "down_proj",
    ],
    lora_alpha = 16,             # LoRA 학습 강도 (보통 r과 같게 설정)
    lora_dropout = 0,            # 뉴런 드롭 비율 (0 = Unsloth 최적화)
    bias = "none",               # 바이어스 학습 여부 ("none" = Unsloth 최적화)
    use_gradient_checkpointing = "unsloth",  # VRAM 절약 방식 ("unsloth" 권장)
    random_state = 3407,         # 재현성을 위한 랜덤 시드
    use_rslora = False,          # Rank Stabilized LoRA 사용 여부
    loftq_config = None,         # LoftQ 양자화 설정 (실습에서는 불필요)
)

print("✅ LoRA 어댑터 추가 완료!")

Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA 어댑터 추가 완료!


# 2. 데이터셋 불러오기

## 1) 데이터셋 로드

In [5]:
from datasets import load_dataset

dataset = load_dataset("teddylee777/QA-Dataset-mini")
dataset

README.md:   0%|          | 0.00/339 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.20k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 16
    })
})

In [6]:
dataset["train"][0]

{'instruction': '바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?',
 'input': '',
 'output': '바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.'}

In [7]:
dataset["train"][1]

{'instruction': 'G7 국가들이 채택한 AI 국제 행동강령에 따르면, 첨단 AI 시스템의 개발 과정에서 어떤 조치를 취해야 합니까?',
 'input': '',
 'output': 'G7 국가들은 첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 평가 및 완화하는 조치를 채택해야 합니다.'}

In [8]:
dataset = dataset["train"]

## 2) formatting_func 만들기

In [9]:
def formatting_prompts_func(data):
    instructions = data["instruction"]  # 리스트로 옴
    outputs = data["output"]            # 리스트로 옴

    result = []
    for instruction, output in zip(instructions, outputs):
        messages = [
            {"role": "user",      "content": instruction},
            {"role": "assistant", "content": output},
        ]
        chat_message = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        result.append(chat_message)

    return result  # 리스트 반환

In [10]:
# Trainer를 만들 때 데이터가 이렇게 정돈됩니다.
data = {
    "instruction": [
        '바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?',
        'G7 국가들이 채택한 AI 국제 행동강령에 따르면, 첨단 AI 시스템의 개발 과정에서 어떤 조치를 취해야 합니까?'
    ],
    "output": [
        '바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.',
        'G7 국가들은 첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 평가 및 완화하는 조치를 채택해야 합니다.'
    ]
}

In [11]:
formatting_prompts_func(data)

['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?<|im_end|>\n<|im_start|>assistant\n바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.<|im_end|>\n',
 '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nG7 국가들이 채택한 AI 국제 행동강령에 따르면, 첨단 AI 시스템의 개발 과정에서 어떤 조치를 취해야 합니까?<|im_end|>\n<|im_start|>assistant\nG7 국가들은 첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 평가 및 완화하는 조치를 채택해야 합니다.<|im_end|>\n']

In [12]:
## 데이터셋 변환 과정
# 데이터셋 원본
{
    'instruction': '바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?',
    'input': '',
    'output': '바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.'
}

# chat형식의 messages로 바꾸기
[
    {"role": "user", "content": '바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?'},
    {"role": "assistant", "content":'바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.'},
]

# 숫자 벡터로 바꾸려면 messages를 하나의 문자열로 변환해야 한다. -- apply_chat_template
'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?<|im_end|>\n<|im_start|>assistant\n바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.<|im_end|>\n'

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?<|im_end|>\n<|im_start|>assistant\n바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.<|im_end|>\n'

# 3. Trainer 만들기

## 1) TrainingArgument

In [13]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

args = TrainingArguments(
    per_device_train_batch_size = 2,         # GPU당 배치 크기
    gradient_accumulation_steps = 4,          # 실질 배치 = 2×4 = 8
    warmup_steps = 5,                         # lr 워밍업 스텝 수
    max_steps = 100,                          # 테스트용 (전체 학습 시 num_train_epochs 사용)
    # num_train_epochs = 3,                   # 전체 학습 시 이걸 사용
    learning_rate = 2e-4,                     # LoRA 권장 학습률
    fp16 = not is_bfloat16_supported(),       # 구형 GPU용
    bf16 = is_bfloat16_supported(),           # 최신 GPU용 (자동 감지)
    logging_steps = 1,                        # 몇 스텝마다 로그 출력할지
    optim = "adamw_8bit",                     # VRAM 절약형 옵티마이저
    weight_decay = 0.01,                      # 과적합 방지 정규화
    lr_scheduler_type = "linear",             # lr 감소 방식
    seed = 3407,                              # 재현성을 위한 랜덤 시드
    output_dir = "outputs",                   # 체크포인트 저장 폴더
    report_to = "none",                       # 로그 기록 위치 (wandb 등)
    average_tokens_across_devices = False,    # 
)

## 2) SFTTrainer

In [14]:
# KeyError가 나면 위에 dataset = dataset["train"] 코드를 추가해주세요``
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    formatting_func = formatting_prompts_func,      
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,               # 전처리 CPU 코어 수
    packing = False,                    # 멀티턴 데이터는 False 권장
    args = args,
)

print("✅ 트레이너 설정 완료!")

Unsloth: Tokenizing ["text"]:   0%|          | 0/16 [00:00<?, ? examples/s]

✅ 트레이너 설정 완료!


# 4. 학습

## 1) 학습 전 GPU 현황

In [15]:
# 학습 전 GPU 메모리 상태 기록 (학습 후 셀과 비교용)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU 이름     : {gpu_stats.name}")
print(f"전체 VRAM    : {max_memory} GB")
print(f"현재 예약량  : {start_gpu_memory} GB")
print(f"남은 여유    : {round(max_memory - start_gpu_memory, 3)} GB")

GPU 이름     : NVIDIA GeForce RTX 4070 Laptop GPU
전체 VRAM    : 7.996 GB
현재 예약량  : 2.949 GB
남은 여유    : 5.047 GB


In [16]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16 | Num Epochs = 50 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,2.927105
2,2.793565
3,2.788972
4,2.692509
5,2.462417
6,2.042781
7,2.087391
8,1.627831
9,1.533561
10,1.498673


In [17]:
# 학습 후 GPU 메모리 및 시간 통계
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"✅ 학습 완료!")
print(f"")
print(f"⏱  학습 시간        : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} 분")
print(f"")
print(f"💾 전체 VRAM 사용량 : {used_memory} GB ({used_percentage} %)")
print(f"💾 LoRA 학습 사용량 : {used_memory_for_lora} GB ({lora_percentage} %)")
print(f"   (모델 로드 제외, 순수 학습에 쓴 VRAM)")

✅ 학습 완료!

⏱  학습 시간        : 4.98 분

💾 전체 VRAM 사용량 : 3.689 GB (46.136 %)
💾 LoRA 학습 사용량 : 0.74 GB (9.255 %)
   (모델 로드 제외, 순수 학습에 쓴 VRAM)


## 3) 모델 저장

In [18]:
# 학습 후 16bit로 병합 저장
model.save_pretrained_merged(
    "models/model_merged_qwen_teddynote",
    tokenizer,
    save_method="merged_16bit"
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: C:\Users\user\.cache\huggingface\hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [02:20<00:00, 140.54s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:06<00:00,  6.06s/it]


Unsloth: Merge process complete. Saved to `c:\potenup3\03-utilizing-huggingface\models\model_merged_qwen_teddynote`


# 5. 추론

In [1]:
# unsloth Bug : 현재 FastLanguageModel로 추론하는 것은 버그 존재
# 이때 model에 unsloth가 반영되어 있기 때문에 재시작 한 후에 불러와야 한다.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ── 병합된 모델 로드 ──────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    "models/model_merged_qwen_teddynote",
    dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("models/model_merged_qwen_teddynote")

print("✅ 모델 로드 완료!")

W0330 11:48:15.854000 43780 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from 'models/model_merged_qwen_teddynote' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ 모델 로드 완료!


In [2]:
input_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "테디노트 유튜브 채널에 대해 알려주세요"}],
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
print(inputs)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198, 130229,  89235, 127121,
          28626, 126310, 144293, 131196,   3315,    109,    226, 139287,  19391,
         131978, 138630,  91669, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [3]:
# AttributeError: 'Qwen2Attention' object has no attribute 'apply_qkv' -> 커널 재시작 후 모델 다시 불러와주세요
# model 속에 unsloth가 반영되어있어 이를 없애기 위해 재시작이 필요합니다.
from transformers import TextStreamer

print("=== 학습 후 출력 (After) ===")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    use_cache=False,
    repetition_penalty=1.3,  
    temperature=0.7,      
    do_sample=True,           
)

=== 학습 후 출력 (After) ===
테디노트(TeddyNote)는 데이터 분석과 머신러닝을 달성하기 위해 사용자 정의 AI 모델로 소프та이어를 교육하는 라벨링 자원 제공 시스템입니다. 이 기업은 미국 텍사스주 알바니아에서 운영되며, 오픈소스 및 자유라이선스 원칙 아래 만들어집니다. 더 많은 정보가 필요하시면 TEDDY 노트 공식 웹사이트를 방문해 주세요: https://teddynote.com/<|im_end|>


# 6. Ollama에 내 모델 등록하기

## 1) Unsloth로 모델 불러오기

In [1]:
from unsloth import FastLanguageModel 
import torch 

max_seq_length = 1024    # 한 번에 처리할 최대 토큰 수 (클수록 VRAM 많이 사용)
dtype = None             # GPU에 맞게 자동 선택 (None 권장)
load_in_4bit = False      # 4bit 압축 로드 여부 (True = VRAM 절약)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "models/model_merged_qwen_teddynote",
    max_seq_length = max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

print(f"✅ 모델 로드 완료!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0330 12:02:05.094000 29456 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4070 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from 'models/model_merged_qwen_teddynote' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'models/model_merged_qwen_teddynote' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ 모델 로드 완료!


## 2) GGUF로 모델 저장하기

In [2]:
# GGUF 변환 + 저장
# cmake installer, visual studio installer, openssl가 자동으로 뜸
# vscode 껏다 켜기
model.save_pretrained_gguf(
    "models/model_merged_qwen_teddynote_gguf",           # 저장 폴더
    tokenizer,
    quantization_method = "q4_k_m"        # 양자화 방식
)

Unsloth: Model is not a PEFT model. Using existing checkpoint at models/model_merged_qwen_teddynote
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Missing packages: openssl
Unsloth: Will attempt to install missing system packages.
Unsloth: Installing openssl via winget (ShiningLight.OpenSSL.Dev)...


Exception in thread Thread-7 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xe2 in position 131: illegal multibyte sequence


Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes


Exception in thread Thread-11 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 48: illegal multibyte sequence


Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['models/model_merged_qwen_teddynote_gguf\\model_merged_qwen_teddynote.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['models/model_merged_qwen_teddynote_gguf\\model_merged_qwen_teddynote.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'models/model_merged_qwen_teddynote'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: C:\Users\user\.unsloth\llama.cpp\build\bin\Release\llama-cli.exe --model models/model_merged_qwen_teddynote_gguf\model_merged_qwen_teddynote.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': 'models/model_merged_qwen_teddynote',
 'gguf_directory': 'models/model_merged_qwen_teddynote_gguf',
 'gguf_files': ['models/model_merged_qwen_teddynote_gguf\\model_merged_qwen_teddynote.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

## 3) Ollama에 내 모델 GGUF 등록하기

In [ ]:
# 터미널에서 실행하세요
# cd models/model_merged_qwen_teddynote_gguf
# ollama create 내모델이름설정 -f Modelfile
# ollama Desktop 켜보세요